In [ ]:
!pip install brainscore-vision
!pip install git+https://github.com/brain-score/brainio.git

In [1]:
from brainscore_vision import load_model
from brainio.stimuli import StimulusSet
from pathlib import Path

# Load a specific BrainScore model
model = load_model("vonegrcnn_47e")

# Now `model` is an object implementing the BrainModel interface
for function in dir(model):
    if not "__" in function:
        print(function)


/Users/domenicbersch/anaconda3/envs/BERG11/lib/python3.11/site-packages/brainscore_core/metrics/__init__.py:16: FutureWarning: xarray subclass Score should explicitly define __slots__
  class Score(DataAssembly):


Neuronal distributions gabor parameters


/Users/domenicbersch/anaconda3/envs/BERG11/lib/python3.11/site-packages/brainscore_vision/models/vonegrcnn_47e/model.py:102: RuntimeWarning: invalid value encountered in divide
  ny_dist_marg = n_joint_dist / n_joint_dist.sum(axis=1, keepdims=True)
/Users/domenicbersch/anaconda3/envs/BERG11/lib/python3.11/site-packages/torch/functional.py:505: UserWarning: torch.meshgrid: in an upcoming release, it will be required to pass the indexing argument. (Triggered internally at /Users/runner/work/pytorch/pytorch/pytorch/aten/src/ATen/native/TensorShape.cpp:4319.)
  return _VF.meshgrid(tensors, **kwargs)  # type: ignore[attr-defined]


RecordingTarget
Task
_logger
_visual_degrees
activations_model
behavior_model
do_behavior
identifier
layer_model
layers
load_region_layer_map_json
look_at
start_recording
start_task
visual_degrees


In [2]:
from brainscore_vision import load_model
from brainio.stimuli import StimulusSet
from pathlib import Path
import pandas as pd

model = load_model("vonegrcnn_47e")
model.start_recording(recording_target=model.RecordingTarget.V1, time_bins=[(100, 200)])

# Get images (filter for valid image files only)
image_paths = sorted(Path("Additionals/testimages/monkeyN_testimages").glob("*.jpg"))
stimulus_ids = [p.stem for p in image_paths]

# Create StimulusSet
stimuli = StimulusSet(pd.DataFrame({
    "stimulus_id": stimulus_ids,
    "filename": [p.name for p in image_paths],
}))
stimuli.stimulus_paths = {sid: str(path.absolute()) for sid, path in zip(stimulus_ids, image_paths)}


layers = list(dict.fromkeys(model.layer_model.region_layer_map.values()))  # Remove duplicates
activations = model.activations_model._extractor.from_stimulus_set(
    stimulus_set=stimuli,
    layers=layers
)

Neuronal distributions gabor parameters


activations:   0%|          | 0/128 [00:00<?, ?it/s]

layer packaging:   0%|          | 0/3 [00:00<?, ?it/s]

In [10]:
activations

<xarray.NeuroidAssembly (presentation: 100, neuroid: 50176)>
array([[-5.89735955e-02, -7.60244280e-02,  1.88235566e-03, ...,
        -3.14954668e-05, -7.50807747e-02, -5.81126884e-02],
       [ 3.74215469e-02, -4.01581787e-02,  1.06550999e-01, ...,
        -1.18112147e-01, -1.48535609e-01, -1.39047265e-01],
       [ 6.46574423e-02,  2.31132265e-02,  2.82968208e-02, ...,
        -4.90820482e-02, -6.57648444e-02, -5.39422557e-02],
       ...,
       [ 2.56092846e-03,  3.46822627e-02,  1.47953965e-02, ...,
        -1.21739775e-01, -1.47582024e-01, -4.29378338e-02],
       [-1.24141097e-01, -6.30976036e-02, -5.56125939e-02, ...,
        -6.97640851e-02, -5.82645424e-02, -1.16804913e-01],
       [ 2.33586431e-02,  3.61305848e-02, -3.03279422e-03, ...,
        -1.17081001e-01, -5.81904352e-02, -2.93788668e-02]], dtype=float32)
Coordinates:
  * neuroid                       (neuroid) MultiIndex
  - neuroid_num                   (neuroid) int64 0 1 2 3 ... 50173 50174 50175
  - model                         (neuroid) object 'vonegrcnn_47e' ... 'voneg...
  - layer                         (neuroid) object 'module.model.layer3' ... ...
  - channel                       (neuroid) int64 0 0 0 0 ... 1023 1023 1023
  - channel_x                     (neuroid) int64 0 0 0 0 0 0 0 ... 6 6 6 6 6 6
  - channel_y                     (neuroid) int64 0 1 2 3 4 5 6 ... 1 2 3 4 5 6
  - neuroid_id                    (neuroid) object 'vonegrcnn_47e.module.mode...
  * presentation                  (presentation) MultiIndex
  - microsaccade_shift_y_degrees  (presentation) float64 0.0 0.0 0.0 ... 0.0 0.0
  - microsaccade_shift_y_pixels   (presentation) float64 0.0 0.0 0.0 ... 0.0 0.0
  - stimulus_id                   (presentation) object '00001_alligator_14n'...
  - microsaccade_shift_x_pixels   (presentation) float64 0.0 0.0 0.0 ... 0.0 0.0
  - microsaccade_shift_x_degrees  (presentation) float64 0.0 0.0 0.0 ... 0.0 0.0
  - filename                      (presentation) object '00001_alligator_14n....

In [7]:
layers


['module.bottleneck', 'module.model.layer2', 'module.model.layer3']

In [8]:
layers = ['module.model.layer3']

activations = model.activations_model._extractor.from_stimulus_set(
    stimulus_set=stimuli,
    layers=layers
)

activations:   0%|          | 0/128 [00:00<?, ?it/s]

layer packaging:   0%|          | 0/1 [00:00<?, ?it/s]

In [9]:
activations.shape

(100, 50176)

# Try it out savely

In [1]:
# Test 1: Discover available BrainScore models
import importlib
import pkgutil

def discover_brainscore_models():
    """Scan brainscore_vision.models package for available models."""
    try:
        import brainscore_vision.models as bs_models
        
        models = []
        # Iterate through all submodules in brainscore_vision.models
        for importer, modname, ispkg in pkgutil.iter_modules(bs_models.__path__):
            if not modname.startswith('_'):  # Skip private modules
                models.append(modname)
        
        return sorted(models)
    except ImportError:
        print("brainscore_vision not installed")
        return []

# Run discovery
available_models = discover_brainscore_models()
print(f"Found {len(available_models)} BrainScore models:")
for i, model in enumerate(available_models[:10], 1):  # Show first 10
    print(f"  {i}. {model}")
if len(available_models) > 10:
    print(f"  ... and {len(available_models) - 10} more")

Found 440 BrainScore models:
  1. AT_efficientnet_b2
  2. AdvProp_efficientnet_b2
  3. AdvProp_efficientnet_b4
  4. AdvProp_efficientnet_b6
  5. AdvProp_efficientnet_b7
  6. AdvProp_efficientnet_b8
  7. AlexNet_SIN
  8. AlexNet_SIN_fov
  9. BiT_S_R101x1
  10. BiT_S_R101x3
  ... and 430 more


/Users/domenicbersch/anaconda3/envs/BERG11/lib/python3.11/site-packages/brainscore_core/metrics/__init__.py:16: FutureWarning: xarray subclass Score should explicitly define __slots__
  class Score(DataAssembly):


In [2]:
# Test 2: Load a specific BrainScore model and inspect structure
from brainscore_vision import load_model

model_name = "vonegrcnn_47e"  # Change to test different models
print(f"\nLoading model: {model_name}")

model = load_model(model_name)

# Inspect model structure
print(f"\nModel type: {type(model)}")
print(f"Has RecordingTarget: {hasattr(model, 'RecordingTarget')}")

if hasattr(model, 'RecordingTarget'):
    recording_targets = [attr for attr in dir(model.RecordingTarget) 
                        if not attr.startswith('_')]
    print(f"Available recording targets: {recording_targets}")

# Check layer structure
if hasattr(model, 'layer_model'):
    print(f"\nLayer model type: {type(model.layer_model)}")
    if hasattr(model.layer_model, 'region_layer_map'):
        print(f"Region-layer map keys: {list(model.layer_model.region_layer_map.keys())}")


Loading model: vonegrcnn_47e
Neuronal distributions gabor parameters


/Users/domenicbersch/anaconda3/envs/BERG11/lib/python3.11/site-packages/brainscore_vision/models/vonegrcnn_47e/model.py:102: RuntimeWarning: invalid value encountered in divide
  ny_dist_marg = n_joint_dist / n_joint_dist.sum(axis=1, keepdims=True)
/Users/domenicbersch/anaconda3/envs/BERG11/lib/python3.11/site-packages/torch/functional.py:505: UserWarning: torch.meshgrid: in an upcoming release, it will be required to pass the indexing argument. (Triggered internally at /Users/runner/work/pytorch/pytorch/pytorch/aten/src/ATen/native/TensorShape.cpp:4319.)
  return _VF.meshgrid(tensors, **kwargs)  # type: ignore[attr-defined]



Model type: <class 'brainscore_vision.model_helpers.brain_transformation.ModelCommitment'>
Has RecordingTarget: True
Available recording targets: ['IT', 'V1', 'V2', 'V4']

Layer model type: <class 'brainscore_vision.model_helpers.brain_transformation.temporal.TemporalAligned'>
Region-layer map keys: ['V1', 'V2', 'V4', 'IT']


In [7]:
# Test 3: Run inference with image paths
from brainscore_vision import load_model
from brainio.stimuli import StimulusSet
from pathlib import Path
import pandas as pd

# Setup
model_name = "vonegrcnn_47e"
image_folder = "/Users/domenicbersch/Documents/Repositories/BERG/Additionals/testimages/monkeyN_testimages"  # Your path
recording_target = "V1"
time_bins = [(100, 200)]

# Load model
model = load_model(model_name)

# Start recording
print(f"Recording from {recording_target} with time bins {time_bins}")
model.start_recording(
    recording_target=getattr(model.RecordingTarget, recording_target),
    time_bins=time_bins
)

# Get image paths
image_paths = sorted(Path(image_folder).glob("*.jpg"))
print(f"Found {len(image_paths)} images")

# Create StimulusSet
stimulus_ids = [p.stem for p in image_paths]
stimuli = StimulusSet(pd.DataFrame({
    "stimulus_id": stimulus_ids,
    "filename": [p.name for p in image_paths],
}))
stimuli.stimulus_paths = {
    sid: str(path.absolute()) 
    for sid, path in zip(stimulus_ids, image_paths)
}

# Extract activations
layers = list(dict.fromkeys(model.layer_model.region_layer_map.values()))
print(f"Extracting from {len(layers)} layers")

activations = model.activations_model._extractor.from_stimulus_set(
    stimulus_set=stimuli,
    layers=layers
)

# Check output
print(f"\nActivations shape: {activations.shape}")
print(f"Activations dtype: {activations.dtype}")
print(f"Sample values: {activations[0, :5]}")

Neuronal distributions gabor parameters


/Users/domenicbersch/anaconda3/envs/BERG11/lib/python3.11/site-packages/brainscore_vision/models/vonegrcnn_47e/model.py:102: RuntimeWarning: invalid value encountered in divide
  ny_dist_marg = n_joint_dist / n_joint_dist.sum(axis=1, keepdims=True)


Recording from V1 with time bins [(100, 200)]
Found 100 images
Extracting from 3 layers


activations:   0%|          | 0/128 [00:00<?, ?it/s]

layer packaging:   0%|          | 0/3 [00:00<?, ?it/s]


Activations shape: (100, 351232)
Activations dtype: float32
Sample values: <xarray.NeuroidAssembly (neuroid: 5)>
array([2.0864072, 3.965712 , 6.820739 , 4.3204846, 4.2370934],
      dtype=float32)
Coordinates:
  * neuroid       (neuroid) MultiIndex
  - neuroid_num   (neuroid) int64 0 1 2 3 4
  - model         (neuroid) object 'vonegrcnn_47e' ... 'vonegrcnn_47e'
  - layer         (neuroid) object 'module.bottleneck' ... 'module.bottleneck'
  - channel       (neuroid) int64 0 0 0 0 0
  - channel_x     (neuroid) int64 0 0 0 0 0
  - channel_y     (neuroid) int64 0 1 2 3 4
  - neuroid_id    (neuroid) object 'vonegrcnn_47e.module.bottleneck.0' ... 'v...
    presentation  object ('00001_alligator_14n.jpg', 0.0, 0.0, '00001_alligat...


In [8]:
# Test 4: Test with different recording targets and models
test_configs = [
    ("vonegrcnn_47e", "V1", [(70, 170)]),
    ("vonegrcnn_47e", "V4", [(100, 200)]),
    # Add more model/region combinations
]

for model_name, region, time_bins in test_configs:
    print(f"\n{'='*60}")
    print(f"Testing: {model_name} | {region} | {time_bins}")
    print('='*60)
    
    try:
        model = load_model(model_name)
        model.start_recording(
            recording_target=getattr(model.RecordingTarget, region),
            time_bins=time_bins
        )
        print(f"✓ Successfully configured {model_name}")
    except Exception as e:
        print(f"✗ Failed: {e}")


Testing: vonegrcnn_47e | V1 | [(70, 170)]
Neuronal distributions gabor parameters


/Users/domenicbersch/anaconda3/envs/BERG11/lib/python3.11/site-packages/brainscore_vision/models/vonegrcnn_47e/model.py:102: RuntimeWarning: invalid value encountered in divide
  ny_dist_marg = n_joint_dist / n_joint_dist.sum(axis=1, keepdims=True)


✓ Successfully configured vonegrcnn_47e

Testing: vonegrcnn_47e | V4 | [(100, 200)]
Neuronal distributions gabor parameters
✓ Successfully configured vonegrcnn_47e


In [12]:
import brainscore_vision

# Check BrainScore's internal cache path
if hasattr(brainscore_vision, '__file__'):
    print(f"BrainScore installed at: {brainscore_vision.__file__}")

# Check if there's a config module
try:
    from brainscore_core import _config
    if hasattr(_config, 'BRAINSCORE_HOME'):
        print(f"Config BRAINSCORE_HOME: {_config.BRAINSCORE_HOME}")
except:
    pass

# List what's actually in the default cache after loading alexnet
from pathlib import Path
cache_dir = Path.home() / ".brain-score"
if cache_dir.exists():
    files = list(cache_dir.rglob("*alexnet*"))
    print(f"Found {len(files)} alexnet-related files")
    if files:
        print(f"Example: {files[0]}")

BrainScore installed at: /Users/domenicbersch/anaconda3/envs/BERG11/lib/python3.11/site-packages/brainscore_vision/__init__.py
Found 0 alexnet-related files


In [1]:
# Add to your Python path if needed
import sys


from berg.models.ephys.brainscore import BrainScoreGateway, discover_brainscore_models

# Test 1: Discovery
models = discover_brainscore_models()
print(f"Found {len(models)} models")
print(models[:5])

# Test 2: Initialize and load
gateway = BrainScoreGateway(
    berg_dir="/your/berg/dir",
    model_id="brainscore-vonegrcnn_47e",
    device="auto",
    selection={'region': 'V1', 'time_bins': [(100, 200)]}
)
gateway.load_model()

# Test 3: Generate responses
responses = gateway.generate_response(
    stimulus="Additionals/testimages/monkeyN_testimages",
    show_progress=True
)
print(f"Response shape: {responses.shape}")

ModuleNotFoundError: No module named 'berg.models.ephys'